# ERCOT Load — Staging Layer (Raw → Cleaned)

This step:
- Removes trailing "DST" labels
- Normalizes utility-style "24:00" to next-day "00:00"
- Parses timestamps safely
- Casts all zone columns to numeric
- Preserves a DST audit flag

Output:
- `ercot.stg_ercot_load`


In [0]:
%sql

CREATE OR REPLACE VIEW ercot.stg_ercot_load AS
WITH base AS (
  SELECT
    hour_ending,

    -- Remove trailing " DST"
    regexp_replace(trim(hour_ending), '\\s*DST\\s*$', '') AS hour_nodst,

    CASE WHEN hour_ending LIKE '%DST%' THEN 1 ELSE 0 END AS is_dst_flag,

    coast, east, fwest, north, ncent, south, scent, west, ercot
  FROM ercot.raw_ercot_hourly_load
  WHERE hour_ending IS NOT NULL
),

norm AS (
  SELECT
    is_dst_flag,

    CASE
      WHEN hour_nodst RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4} 24:00$' THEN
        date_format(
          date_add(to_date(substr(hour_nodst, 1, 10), 'MM/dd/yyyy'), 1),
          'MM/dd/yyyy'
        ) || ' 00:00'
      ELSE hour_nodst
    END AS hour_norm,

    coast, east, fwest, north, ncent, south, scent, west, ercot
  FROM base
),

parsed AS (
  SELECT
    is_dst_flag,

    COALESCE(
      try_to_timestamp(hour_norm, 'MM/dd/yyyy HH:mm'),
      try_to_timestamp(hour_norm, 'yyyy-MM-dd HH:mm:ss'),
      try_to_timestamp(hour_norm, 'yyyy-MM-dd HH:mm')
    ) AS hour_ts,

    CAST(coast AS DOUBLE) AS coast_mw,
    CAST(east  AS DOUBLE) AS east_mw,
    CAST(fwest AS DOUBLE) AS fwest_mw,
    CAST(north AS DOUBLE) AS north_mw,
    CAST(ncent AS DOUBLE) AS ncent_mw,
    CAST(south AS DOUBLE) AS south_mw,
    CAST(scent AS DOUBLE) AS scent_mw,
    CAST(west  AS DOUBLE) AS west_mw,
    CAST(ercot AS DOUBLE) AS ercot_mw,

    hour_norm AS hour_raw_clean
  FROM norm
)

SELECT *
FROM parsed
WHERE hour_ts IS NOT NULL;


# Hourly Normalization Layer

Goal:
- Ensure exactly one row per hour
- Average duplicate DST hours
- Produce stable hourly time series

Output:
- `ercot.stg_ercot_load_1h`


In [0]:
%sql

CREATE OR REPLACE VIEW ercot.stg_ercot_load_1h AS
SELECT
  hour_ts,
  AVG(ercot_mw) AS ercot_mw,
  AVG(coast_mw) AS coast_mw,
  AVG(east_mw)  AS east_mw,
  AVG(fwest_mw) AS fwest_mw,
  AVG(north_mw) AS north_mw,
  AVG(ncent_mw) AS ncent_mw,
  AVG(south_mw) AS south_mw,
  AVG(scent_mw) AS scent_mw,
  AVG(west_mw)  AS west_mw
FROM ercot.stg_ercot_load
GROUP BY hour_ts;


# Data Quality Checks

Sanity validation:
- No null timestamps
- Exactly one row per hour
- No negative MW values



In [0]:
%sql

-- Row count
SELECT COUNT(*) AS total_rows
FROM ercot.stg_ercot_load_1h;

-- Duplicate hour check
SELECT hour_ts, COUNT(*) AS cnt
FROM ercot.stg_ercot_load_1h
GROUP BY hour_ts
HAVING COUNT(*) > 1;

-- Negative load check
SELECT COUNT(*) AS negative_values
FROM ercot.stg_ercot_load_1h
WHERE ercot_mw < 0;


# Databricks notebook source


In [0]:
df = spark.table("ercot.stg_ercot_load_1h")

pdf = df.toPandas()

import base64

csv_text = pdf.to_csv(index=False)
b64 = base64.b64encode(csv_text.encode()).decode()

html = f"""
<a download="ercot_load_1h.csv"
   href="data:text/csv;base64,{b64}"
   style="font-size:16px; font-weight:600;">
   ⬇️ Download ercot_load_1h.csv
</a>
"""
displayHTML(html)


After export:

Download the generated CSV from:

https://community.cloud.databricks.com/files/ercot_export/ercot_load_1h/

Rename to:

    ercot_load_1h.csv

Place inside your local dbt repo:

    seeds/ercot_load_1h.csv
